<a href="https://colab.research.google.com/github/DokPan/LabWorks/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22MPIT_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""🎯 СИСТЕМА РЕКОМЕНДАЦИЙ ЦЕН - ОБУЧЕНИЕ МОДЕЛИ"""

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from google.colab import files
import io
import joblib
import warnings
warnings.filterwarnings('ignore')

class PriceRecommender:
    """Полнофункциональный класс рекомендаций цен"""

    def __init__(self):
        self.model = None
        self.feature_columns = []
        self.passenger_max_ratio = 1.3

    def preprocess_data(self, df):
        """Предобработка данных"""
        print("🔧 Предобработка данных...")

        df_processed = df.copy()

        # Валидация обязательных колонок
        required_columns = ['is_done', 'distance_in_meters', 'duration_in_seconds', 'order_timestamp']
        missing_columns = [col for col in required_columns if col not in df_processed.columns]
        if missing_columns:
            raise ValueError(f"❌ Отсутствуют колонки: {missing_columns}")

        # Целевая переменная
        df_processed['is_done_numeric'] = (df_processed['is_done'] == 'done').astype(int)
        print(f"   ✅ Принятые заказы: {(df_processed['is_done_numeric'] == 1).sum()}")

        # Преобразование единиц
        df_processed['distance_km'] = df_processed['distance_in_meters'] / 1000.0
        df_processed['duration_min'] = df_processed['duration_in_seconds'] / 60.0

        # Временные признаки
        df_processed['order_datetime'] = pd.to_datetime(df_processed['order_timestamp'])
        df_processed['order_hour'] = df_processed['order_datetime'].dt.hour
        df_processed['order_day_of_week'] = df_processed['order_datetime'].dt.dayofweek
        df_processed['order_month'] = df_processed['order_datetime'].dt.month

        # Сезонность
        def get_season(month):
            if month in [12, 1, 2]: return 'winter'
            elif month in [3, 4, 5]: return 'spring'
            elif month in [6, 7, 8]: return 'summer'
            else: return 'autumn'

        df_processed['season'] = df_processed['order_month'].apply(get_season)
        season_map = {'winter': 0, 'spring': 1, 'summer': 2, 'autumn': 3}
        df_processed['season_code'] = df_processed['season'].map(season_map)

        # Погода
        def get_weather(season, hour):
            if season == 'winter': return 'snowy' if hour < 12 else 'cloudy'
            elif season == 'summer': return 'sunny' if hour < 18 else 'rainy'
            else: return 'cloudy' if hour < 12 else 'rainy'

        df_processed['weather_type'] = [
            get_weather(s, h) for s, h in zip(df_processed['season'], df_processed['order_hour'])
        ]
        weather_map = {'sunny': 0, 'cloudy': 1, 'rainy': 2, 'snowy': 3}
        df_processed['weather_code'] = df_processed['weather_type'].map(weather_map)
        df_processed['bad_weather'] = (df_processed['weather_code'] >= 2).astype(int)

        # Временные признаки
        df_processed['is_weekend'] = (df_processed['order_day_of_week'] >= 5).astype(int)
        df_processed['is_peak_hours'] = (
            ((df_processed['order_hour'] >= 7) & (df_processed['order_hour'] <= 10) |
             (df_processed['order_hour'] >= 17) & (df_processed['order_hour'] <= 20)) &
            (df_processed['is_weekend'] == 0)
        ).astype(int)

        print("✅ Данные обработаны")
        return df_processed

    def train_model(self, df):
        """Обучение модели"""
        print("💰 Обучение модели...")

        accepted_orders = df[df['is_done_numeric'] == 1]
        if len(accepted_orders) == 0:
            raise ValueError("❌ Нет принятых заказов для обучения")

        # Признаки модели
        self.feature_columns = [
            'distance_km', 'duration_min', 'order_hour', 'is_weekend',
            'is_peak_hours', 'bad_weather', 'season_code', 'weather_code'
        ]

        X = accepted_orders[self.feature_columns]
        y = accepted_orders['price_bid_local']

        # Обучение
        self.model = HistGradientBoostingRegressor(
            random_state=42, max_iter=100, learning_rate=0.1
        )
        self.model.fit(X, y)

        # Предсказание минимальных цен
        df['min_driver_price'] = self.model.predict(df[self.feature_columns])
        df['min_driver_price'] = np.round(df['min_driver_price'] / 10) * 10
        df['min_driver_price'] = np.maximum(df['min_driver_price'], df['price_start_local'] * 0.9)

        print(f"✅ Модель обучена на {len(accepted_orders)} заказах")
        return df

    def analyze_behavior(self, df):
        """Анализ поведения пассажиров"""
        print("📊 Анализ поведения...")

        df_analysis = df.copy()
        df_analysis['price_ratio'] = df_analysis['price_bid_local'] / df_analysis['price_start_local']

        accepted_ratios = df_analysis[df_analysis['is_done_numeric'] == 1]['price_ratio']

        if len(accepted_ratios) > 0:
            self.passenger_max_ratio = np.percentile(accepted_ratios, 75)
            self.passenger_max_ratio = min(self.passenger_max_ratio, 1.5)
            self.passenger_max_ratio = max(self.passenger_max_ratio, 1.2)

        print(f"🎯 Максимальное соотношение: {self.passenger_max_ratio:.2f}")
        return self.passenger_max_ratio

    def _calculate_dynamic_max_ratio(self, order):
        """Расчет динамического максимального соотношения"""
        base_ratio = self.passenger_max_ratio

        passenger_price = order['price_start_local']
        if passenger_price > 3000: base_ratio *= 1.15
        elif passenger_price > 2000: base_ratio *= 1.10
        elif passenger_price > 1000: base_ratio *= 1.05

        distance_km = order.get('distance_km', 5)
        if distance_km > 15: base_ratio *= 1.12
        elif distance_km > 10: base_ratio *= 1.08

        duration_min = order.get('duration_min', 15)
        if duration_min > 40: base_ratio *= 1.10
        elif duration_min > 25: base_ratio *= 1.06

        return min(base_ratio, 1.8)

    def _calculate_weights(self, order):
        """Расчет весов для формулы цены"""
        min_weight, max_weight = 0.4, 0.6

        passenger_price = order['price_start_local']
        if passenger_price > 2500: min_weight, max_weight = 0.5, 0.5
        elif passenger_price > 1500: min_weight, max_weight = 0.45, 0.55

        if order.get('is_peak_hours', 0) == 1:
            min_weight += 0.1
            max_weight -= 0.1

        if order.get('distance_km', 5) < 3:
            min_weight -= 0.05
            max_weight += 0.05

        return min_weight, max_weight

    def _calculate_increase_limits(self, order):
        """Расчет ограничений повышения цены"""
        min_increase, max_increase = 1.03, 1.50

        passenger_price = order['price_start_local']
        if passenger_price > 3000: min_increase, max_increase = 1.08, 1.35
        elif passenger_price > 2000: min_increase, max_increase = 1.06, 1.40
        elif passenger_price > 1000: min_increase, max_increase = 1.04, 1.45

        distance_km = order.get('distance_km', 5)
        if distance_km < 3: max_increase = 1.25
        elif distance_km < 5: max_increase = 1.35

        if order.get('is_peak_hours', 0) == 1:
            min_increase = max(min_increase, 1.08)

        return min_increase, max_increase

    def _calculate_price_multiplier(self, order):
        """Расчет множителя цены"""
        multiplier = 1.0

        if order.get('is_peak_hours', 0) == 1: multiplier *= 1.12
        if order.get('bad_weather', 0) == 1: multiplier *= 1.10
        if order.get('is_weekend', 0) == 1: multiplier *= 1.06

        hour = order.get('order_hour', 12)
        if 18 <= hour <= 23: multiplier *= 1.08
        elif 0 <= hour <= 6: multiplier *= 1.12

        distance_km = order.get('distance_km', 5)
        if distance_km > 20: multiplier *= 1.15
        elif distance_km > 15: multiplier *= 1.10
        elif distance_km > 10: multiplier *= 1.06
        elif distance_km < 3: multiplier *= 0.95

        duration_min = order.get('duration_min', 15)
        if duration_min > 45: multiplier *= 1.10
        elif duration_min > 30: multiplier *= 1.06

        return multiplier

    def _calculate_accept_probability(self, price_ratio):
        """Расчет вероятности принятия"""
        if price_ratio <= 1.05: return 0.85
        elif price_ratio <= 1.08: return 0.80
        elif price_ratio <= 1.12: return 0.75
        elif price_ratio <= 1.16: return 0.65
        elif price_ratio <= 1.20: return 0.55
        elif price_ratio <= 1.25: return 0.45
        elif price_ratio <= 1.30: return 0.35
        elif price_ratio <= 1.40: return 0.25
        else: return 0.15

    def recommend_price(self, order):
        """Рекомендация цены для заказа - ОСНОВНАЯ ЛОГИКА"""
        if self.model is None:
            raise ValueError("❌ Модель не обучена")

        passenger_price = float(order['price_start_local'])

        # Предсказание модели
        features = {}
        for feature in self.feature_columns:
            features[feature] = order.get(feature, 0)

        feature_vector = pd.DataFrame([features])[self.feature_columns]
        min_driver_price = self.model.predict(feature_vector)[0]

        # Гарантия минимальной цены
        min_driver_price = max(min_driver_price, passenger_price * 0.95)

        # Расчет цены
        base_price = max(passenger_price, min_driver_price)
        multiplier = self._calculate_price_multiplier(order)
        adjusted_min_price = base_price * multiplier

        dynamic_max_ratio = self._calculate_dynamic_max_ratio(order)
        max_passenger_price = passenger_price * dynamic_max_ratio

        min_weight, max_weight = self._calculate_weights(order)
        recommended_price = (adjusted_min_price * min_weight + max_passenger_price * max_weight)

        # Применение ограничений
        min_increase, max_increase = self._calculate_increase_limits(order)
        recommended_price = max(recommended_price, passenger_price * min_increase)
        recommended_price = min(recommended_price, passenger_price * max_increase)

        # Округление
        last_digit = recommended_price % 10
        if last_digit not in [0, 5]:
            recommended_price = round(recommended_price / 10) * 10

        # Расчет вероятности
        price_ratio = recommended_price / passenger_price
        probability = self._calculate_accept_probability(price_ratio)

        price_increase_percent = ((recommended_price - passenger_price) / passenger_price * 100)

        return {
            'order_id': order.get('order_id', 'unknown'),
            'passenger_price': passenger_price,
            'recommended_price': int(recommended_price),
            'price_increase_percent': round(price_increase_percent, 1),
            'accept_probability': round(probability, 3)
        }

    def recommend_prices_batch(self, df, sample_size=50):
        """Пакетная рекомендация цен"""
        print(f"🎯 Генерация рекомендаций для {sample_size} заказов...")

        df_sample = df.sample(n=min(sample_size, len(df)), random_state=42)

        recommendations = []
        for idx, (_, order) in enumerate(df_sample.iterrows()):
            try:
                recommendation = self.recommend_price(order.to_dict())
                recommendations.append(recommendation)
            except Exception as e:
                print(f"⚠️ Ошибка в заказе {idx}: {e}")
                continue

        print(f"✅ Сгенерировано {len(recommendations)} рекомендаций")
        return pd.DataFrame(recommendations)

    def save_model(self, filename='price_recommender_model.pkl'):
        """Сохранение обученной модели"""
        model_data = {
            'model': self.model,
            'feature_columns': self.feature_columns,
            'passenger_max_ratio': self.passenger_max_ratio,
            'model_type': 'PriceRecommender'
        }

        joblib.dump(model_data, filename)
        print(f"💾 Модель сохранена как '{filename}'")

# 🔥 ФУНКЦИИ ДЛЯ ИНТЕРФЕЙСА - вызывают логику модели
def load_recommender_model(filename='price_recommender_model.pkl'):
    """Загрузка модели и создание рекомендатора"""
    try:
        model_data = joblib.load(filename)

        # Создаем объект рекомендатора
        recommender = PriceRecommender()
        recommender.model = model_data['model']
        recommender.feature_columns = model_data['feature_columns']
        recommender.passenger_max_ratio = model_data['passenger_max_ratio']

        return recommender
    except Exception as e:
        raise ValueError(f"Ошибка загрузки модели: {e}")

def recommend_price_from_model(order, model_filename='price_recommender_model.pkl'):
    """Упрощенная функция для интерфейса"""
    recommender = load_recommender_model(model_filename)
    return recommender.recommend_price(order)

def main():
    """Основная функция обучения"""
    print("=" * 60)
    print("🚀 ОБУЧЕНИЕ МОДЕЛИ РЕКОМЕНДАЦИЙ ЦЕН")
    print("=" * 60)

    recommender = PriceRecommender()

    try:
        # Загрузка данных
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("❌ Файл не загружен")

        file_name = list(uploaded.keys())[0]
        print(f"📁 Загружен файл: {file_name}")
        df = pd.read_csv(io.BytesIO(uploaded[file_name]))
        print(f"📊 Загружено {len(df)} заказов")

        # Обработка и обучение
        df_processed = recommender.preprocess_data(df)
        df_with_prices = recommender.train_model(df_processed)
        recommender.analyze_behavior(df_with_prices)

        # Тестирование
        print(f"\n💰 ТЕСТИРОВАНИЕ МОДЕЛИ")
        print("=" * 50)

        test_orders = df_with_prices.head(3)
        for idx, (_, order) in enumerate(test_orders.iterrows()):
            try:
                recommendation = recommender.recommend_price(order.to_dict())
                print(f"📋 Заказ {idx+1}: {order['price_start_local']} → {recommendation['recommended_price']} руб. (+{recommendation['price_increase_percent']}%)")
            except Exception as e:
                print(f"⚠️ Ошибка теста {idx+1}: {e}")

        # Сохранение модели
        recommender.save_model('price_recommender_model.pkl')
        print("\n✅ Модель готова к использованию!")

        return recommender

    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return None

if __name__ == "__main__":
    main()

🚀 ОБУЧЕНИЕ МОДЕЛИ РЕКОМЕНДАЦИЙ ЦЕН


Saving test.csv to test.csv
📁 Загружен файл: test.csv
📊 Загружено 50 заказов
🔧 Предобработка данных...
   ✅ Принятые заказы: 20
✅ Данные обработаны
💰 Обучение модели...
✅ Модель обучена на 20 заказах
📊 Анализ поведения...
🎯 Максимальное соотношение: 1.20

💰 ТЕСТИРОВАНИЕ МОДЕЛИ
📋 Заказ 1: 3950 → 5070 руб. (+28.4%)
📋 Заказ 2: 2300 → 2980 руб. (+29.6%)
📋 Заказ 3: 3000 → 4200 руб. (+40.0%)
💾 Модель сохранена как 'price_recommender_model.pkl'

✅ Модель готова к использованию!
